# Sensitivity Plot (portable, auto-build summaries)

This notebook resolves the sensitivity output folder, auto-builds the summary CSVs if they are missing, and then generates the plots.

In [ ]:

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.
import numpy as np

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))


In [ ]:

CONFIG = {
    "sensitivity_output_root": "Output files (Sensitivity, No Mutation, Verified)",
    "manifest_file": "Sensitivity_Case_Manifest.csv",
    "run_status_file": "Sensitivity_Case_Run_Status.csv",
    "case_summary_file": "Sensitivity_Case_Level_Summary.csv",
    "change_summary_file": "Sensitivity_Change_Summary_By_Case.csv",
    "compare_file": "Sensitivity_Change_vs_Baseline.csv",
    "threshold_file": "Sensitivity_First_Change_Thresholds.csv",
    "long_file": "Sensitivity_Match_Level_Long.csv",
    "solution_file_name": "Simulation_Best_Solutions_All_Matches.csv",
    "plot_summary_file_name": "Simulation_Plot_Data_All_Matches.csv",
    "baseline_case_id": "BASE",
    "save_figures": True,
    "auto_build_missing_summaries": True,
}
CONFIG


In [ ]:

FILTERS = {
    "case_families": None,     # Example: ["SellerRisk", "Gamma"]
    "case_ids": None,          # Example: ["BASE", "LS_0253"]
    "match_ids": None,         # Example: [1, 2, 3]
    "include_baseline": True,
}
FILTERS


In [ ]:

def unique_paths(paths):
    out = []
    seen = set()
    for path in paths:
        if path is None:
            continue
        p = Path(path).expanduser()
        try:
            key = str(p.resolve()) if p.exists() else str(p)
        except Exception:
            key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def candidate_roots(*anchors, max_parent_depth: int = 5):
    roots = []
    for anchor in anchors:
        if anchor is None:
            continue
        p = Path(anchor).expanduser()
        if p.suffix:
            p = p.parent
        roots.append(p)
        roots.extend(list(p.parents)[:max_parent_depth])
    return unique_paths(roots)


def resolve_output_root(raw_path, required_files=()):
    p = Path(raw_path).expanduser()
    checked = []
    search_roots = candidate_roots(Path.cwd().resolve(), Path('/mnt/data'))

    def has_required(folder: Path) -> bool:
        return all((folder / f).exists() for f in required_files)

    candidates = []

    if p.is_absolute():
        candidates.append(p)
    else:
        for root in search_roots:
            candidates.append(root / p)
        for root in search_roots:
            if root.exists() and root.is_dir():
                for child in root.iterdir():
                    if child.is_dir():
                        candidates.append(child / p)

    for cand in unique_paths(candidates):
        checked.append(cand)
        if required_files and has_required(cand):
            return cand.resolve()

    for cand in unique_paths(candidates):
        checked.append(cand)
        if cand.exists() and cand.is_dir():
            return cand.resolve()

    raise FileNotFoundError(
        "Could not resolve the sensitivity output root. Tried:\n" + "\n".join(str(x) for x in unique_paths(checked))
    )


def load_case_outputs(case_row: pd.Series, solution_file_name: str, plot_summary_file_name: str):
    case_output_dir = Path(case_row["case_output_dir"])
    solution_path = case_output_dir / solution_file_name
    plot_path = case_output_dir / plot_summary_file_name

    if not solution_path.exists():
        raise FileNotFoundError(f"Missing solution file: {solution_path}")

    solution_df = read_csv_optimized(solution_path)
    solution_df["case_id"] = case_row["case_id"]
    solution_df["case_label"] = case_row["case_label"]
    solution_df["case_family"] = case_row.get("case_family", "")
    solution_df["case_order"] = case_row.get("case_order", np.nan)
    solution_df["case_value"] = case_row.get("case_value", "")
    solution_df["lambda_s"] = case_row.get("lambda_s", np.nan)
    solution_df["lambda_b"] = case_row.get("lambda_b", np.nan)
    solution_df["gamma"] = case_row.get("gamma", np.nan)
    solution_df["volume_resid_multiplier"] = case_row.get("volume_resid_multiplier", np.nan)
    solution_df["price_resid_multiplier"] = case_row.get("price_resid_multiplier", np.nan)

    if plot_path.exists():
        plot_df = read_csv_optimized(plot_path)
        keep_cols = [c for c in [
            "match_id",
            "shape_correlation",
            "basis_correlation",
            "shape_corr_category",
            "basis_corr_category",
            "mean_generation",
            "median_generation",
            "mean_demand",
            "median_demand",
            "mean_seller_price",
            "median_seller_price",
            "mean_buyer_price",
            "median_buyer_price",
            "generation_vs_demand_mean_category",
            "generation_vs_demand_median_category",
            "seller_vs_buyer_price_mean_category",
            "seller_vs_buyer_price_median_category",
        ] if c in plot_df.columns]
        if keep_cols:
            solution_df = solution_df.merge(
                plot_df[keep_cols].drop_duplicates(subset=["match_id"]),
                on="match_id",
                how="left",
            )
    return solution_df


def build_missing_summary_files(output_root: Path):
    manifest_path = output_root / CONFIG["manifest_file"]
    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Manifest file not found at {manifest_path}. "
            f"Run the sensitivity Run notebook first."
        )

    manifest_df = read_csv_optimized(manifest_path)
    status_col = "status" if "status" in manifest_df.columns else None
    if status_col is not None:
        manifest_df = manifest_df.loc[manifest_df[status_col].isin(["success", "skipped_existing_output"])].copy()
    manifest_df = manifest_df.sort_values(["case_order", "case_id"]).reset_index(drop=True)

    if manifest_df.empty:
        raise ValueError("The manifest contains no successful cases to summarize.")

    all_rows = []
    for _, case_row in manifest_df.iterrows():
        case_df = load_case_outputs(
            case_row=case_row,
            solution_file_name=CONFIG["solution_file_name"],
            plot_summary_file_name=CONFIG["plot_summary_file_name"],
        )
        all_rows.append(case_df)

    long_df = pd.concat(all_rows, ignore_index=True)
    long_df["has_ppa"] = long_df["ppa_type"].astype(str).ne("No Contract")
    long_df["is_physical"] = long_df["ppa_type"].astype(str).eq("Physical")
    long_df["is_virtual"] = long_df["ppa_type"].astype(str).eq("Virtual")
    long_df["is_fix"] = long_df["profile_type"].astype(str).eq("Fix")
    long_df["is_asg"] = long_df["profile_type"].astype(str).eq("AsG")
    long_df["is_asc"] = long_df["profile_type"].astype(str).eq("AsC")
    long_df["case_value_numeric"] = pd.to_numeric(long_df["case_value"], errors="coerce")

    long_path = output_root / CONFIG["long_file"]
    long_df.to_csv(long_path, index=False)

    case_summary_df = (
        long_df.groupby(
            ["case_id", "case_label", "case_family", "case_order", "case_value", "case_value_numeric",
             "lambda_s", "lambda_b", "gamma", "volume_resid_multiplier", "price_resid_multiplier"],
            dropna=False,
            observed=False,
        )
        .agg(
            n_matches=("match_id", "nunique"),
            ppa_share=("has_ppa", "mean"),
            physical_share=("is_physical", "mean"),
            virtual_share=("is_virtual", "mean"),
            fix_share=("is_fix", "mean"),
            asg_share=("is_asg", "mean"),
            asc_share=("is_asc", "mean"),
            seller_utility_mean=("seller_utility", "mean"),
            seller_utility_median=("seller_utility", "median"),
            buyer_utility_mean=("buyer_utility", "mean"),
            buyer_utility_median=("buyer_utility", "median"),
            strike_mean=("strike_price_mwh", "mean"),
            strike_median=("strike_price_mwh", "median"),
            volume_mean=("volume_mw", "mean"),
            volume_median=("volume_mw", "median"),
        )
        .reset_index()
        .sort_values(["case_order", "case_id"])
        .reset_index(drop=True)
    )
    case_summary_path = output_root / CONFIG["case_summary_file"]
    case_summary_df.to_csv(case_summary_path, index=False)

    baseline_df = long_df.loc[long_df["case_id"].astype(str) == str(CONFIG["baseline_case_id"])].copy()
    if baseline_df.empty:
        raise ValueError(f"No baseline case found for case_id={CONFIG['baseline_case_id']}")

    baseline_cols = [
        "match_id",
        "ppa_type",
        "profile_type",
        "volume_mw",
        "strike_price_mwh",
        "seller_utility",
        "buyer_utility",
        "has_ppa",
    ]
    baseline_df = baseline_df[baseline_cols].rename(columns={
        "ppa_type": "baseline_ppa_type",
        "profile_type": "baseline_profile_type",
        "volume_mw": "baseline_volume_mw",
        "strike_price_mwh": "baseline_strike_price_mwh",
        "seller_utility": "baseline_seller_utility",
        "buyer_utility": "baseline_buyer_utility",
        "has_ppa": "baseline_has_ppa",
    })

    compare_df = long_df.merge(baseline_df, on="match_id", how="left")
    compare_df["same_ppa_type"] = compare_df["ppa_type"].astype(str).eq(compare_df["baseline_ppa_type"].astype(str))
    compare_df["same_profile_type"] = compare_df["profile_type"].astype(str).eq(compare_df["baseline_profile_type"].astype(str))
    compare_df["same_volume_solution"] = np.isclose(
        pd.to_numeric(compare_df["volume_mw"], errors="coerce"),
        pd.to_numeric(compare_df["baseline_volume_mw"], errors="coerce"),
        equal_nan=True,
        atol=1e-9,
    )
    compare_df["same_price_solution"] = np.isclose(
        pd.to_numeric(compare_df["strike_price_mwh"], errors="coerce"),
        pd.to_numeric(compare_df["baseline_strike_price_mwh"], errors="coerce"),
        equal_nan=True,
        atol=1e-9,
    )
    compare_df["same_full_decision"] = (
        compare_df["same_ppa_type"] &
        compare_df["same_profile_type"] &
        compare_df["same_volume_solution"] &
        compare_df["same_price_solution"]
    )
    compare_df["seller_utility_diff_vs_baseline"] = (
        pd.to_numeric(compare_df["seller_utility"], errors="coerce") -
        pd.to_numeric(compare_df["baseline_seller_utility"], errors="coerce")
    )
    compare_df["buyer_utility_diff_vs_baseline"] = (
        pd.to_numeric(compare_df["buyer_utility"], errors="coerce") -
        pd.to_numeric(compare_df["baseline_buyer_utility"], errors="coerce")
    )
    compare_path = output_root / CONFIG["compare_file"]
    compare_df.to_csv(compare_path, index=False)

    threshold_rows = []
    non_baseline = compare_df.loc[compare_df["case_id"].astype(str) != str(CONFIG["baseline_case_id"])].copy()

    for (case_family, match_id), sub in non_baseline.groupby(["case_family", "match_id"], dropna=False, observed=False):
        ordered = sub.sort_values(["case_order", "case_id"]).reset_index(drop=True)
        changed = ordered.loc[ordered["same_full_decision"] == False].copy()
        if changed.empty:
            threshold_rows.append({
                "case_family": case_family,
                "match_id": match_id,
                "first_change_case_id": "",
                "first_change_case_label": "",
                "first_change_case_order": np.nan,
                "first_change_case_value": "",
            })
        else:
            first = changed.iloc[0]
            threshold_rows.append({
                "case_family": case_family,
                "match_id": match_id,
                "first_change_case_id": first["case_id"],
                "first_change_case_label": first["case_label"],
                "first_change_case_order": first["case_order"],
                "first_change_case_value": first["case_value"],
            })

    threshold_df = pd.DataFrame(threshold_rows).sort_values(["case_family", "match_id"]).reset_index(drop=True)
    threshold_path = output_root / CONFIG["threshold_file"]
    threshold_df.to_csv(threshold_path, index=False)

    change_summary_df = (
        compare_df.groupby(["case_id", "case_label", "case_family", "case_order", "case_value", "case_value_numeric"], dropna=False, observed=False)
        .agg(
            n_matches=("match_id", "nunique"),
            share_same_full_decision=("same_full_decision", "mean"),
            share_same_ppa_type=("same_ppa_type", "mean"),
            share_same_volume=("same_volume_solution", "mean"),
            share_same_price=("same_price_solution", "mean"),
            mean_seller_utility_diff_vs_baseline=("seller_utility_diff_vs_baseline", "mean"),
            mean_buyer_utility_diff_vs_baseline=("buyer_utility_diff_vs_baseline", "mean"),
        )
        .reset_index()
        .sort_values(["case_order", "case_id"])
        .reset_index(drop=True)
    )
    change_summary_path = output_root / CONFIG["change_summary_file"]
    change_summary_df.to_csv(change_summary_path, index=False)

    return {
        "long_df": long_df,
        "case_summary_df": case_summary_df,
        "compare_df": compare_df,
        "change_summary_df": change_summary_df,
        "threshold_df": threshold_df,
        "manifest_df": manifest_df,
    }


def ensure_summary_tables(output_root: Path):
    required = [
        CONFIG["case_summary_file"],
        CONFIG["change_summary_file"],
        CONFIG["compare_file"],
    ]
    if all((output_root / f).exists() for f in required):
        return {
            "case_summary_df": read_csv_optimized(output_root / CONFIG["case_summary_file"]),
            "change_summary_df": read_csv_optimized(output_root / CONFIG["change_summary_file"]),
            "compare_df": read_csv_optimized(output_root / CONFIG["compare_file"]),
        }

    if not CONFIG["auto_build_missing_summaries"]:
        missing = [f for f in required if not (output_root / f).exists()]
        raise FileNotFoundError(
            "Missing required summary files in the sensitivity output folder: "
            + ", ".join(missing)
            + ". Run the Summarize notebook first or set auto_build_missing_summaries=True."
        )

    print("Summary files are missing. Building them now from the manifest and case outputs...")
    built = build_missing_summary_files(output_root)
    print("Built summary files:")
    print("  ", output_root / CONFIG["case_summary_file"])
    print("  ", output_root / CONFIG["change_summary_file"])
    print("  ", output_root / CONFIG["compare_file"])
    return {
        "case_summary_df": built["case_summary_df"],
        "change_summary_df": built["change_summary_df"],
        "compare_df": built["compare_df"],
    }


output_root = resolve_output_root(
    CONFIG["sensitivity_output_root"],
    required_files=[CONFIG["manifest_file"]],
)
tables = ensure_summary_tables(output_root)

case_summary_df = tables["case_summary_df"].copy()
change_summary_df = tables["change_summary_df"].copy()
compare_df = tables["compare_df"].copy()

fig_root = output_root / "Figures"
fig_root.mkdir(parents=True, exist_ok=True)

print("Resolved output root:", output_root)
print("Figure output root:", fig_root)
print("Rows loaded:")
print("  case_summary_df:", len(case_summary_df))
print("  change_summary_df:", len(change_summary_df))
print("  compare_df:", len(compare_df))


In [ ]:

def x_values(df: pd.DataFrame):
    x = pd.to_numeric(df["case_value_numeric"], errors="coerce")
    if x.notna().any():
        return x
    return pd.to_numeric(df["case_order"], errors="coerce")


def apply_case_filters(df: pd.DataFrame):
    out = df.copy()
    if FILTERS["case_families"] is not None and "case_family" in out.columns:
        out = out[out["case_family"].isin(FILTERS["case_families"])].copy()
    if FILTERS["case_ids"] is not None and "case_id" in out.columns:
        out = out[out["case_id"].astype(str).isin([str(x) for x in FILTERS["case_ids"]])].copy()
    if not FILTERS["include_baseline"] and "case_id" in out.columns:
        out = out[out["case_id"].astype(str) != str(CONFIG["baseline_case_id"])].copy()
    return out


def apply_match_filters(df: pd.DataFrame):
    out = apply_case_filters(df)
    if FILTERS["match_ids"] is not None and "match_id" in out.columns:
        out = out[out["match_id"].isin(FILTERS["match_ids"])].copy()
    return out


def savefig_if_needed(fig, filename: str):
    if CONFIG["save_figures"]:
        path = fig_root / filename
        fig.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Saved: {path}")


In [ ]:

case_summary_df = apply_case_filters(case_summary_df)
change_summary_df = apply_case_filters(change_summary_df)
compare_df = apply_match_filters(compare_df)

print("Filtered rows:")
print("  case_summary_df:", len(case_summary_df))
print("  change_summary_df:", len(change_summary_df))
print("  compare_df:", len(compare_df))


In [ ]:

for family, sub in case_summary_df.groupby("case_family", dropna=False, observed=False):
    sub = sub.sort_values(["case_order", "case_id"]).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x_values(sub), sub["ppa_share"], marker="o")
    ax.set_title(f"PPA share by case — {family}")
    ax.set_xlabel("Case value")
    ax.set_ylabel("PPA share")
    ax.grid(True, alpha=0.3)
    savefig_if_needed(fig, f"PPA_share__{family}.png")
    plt.show()


In [ ]:

for family, sub in change_summary_df.groupby("case_family", dropna=False, observed=False):
    sub = sub.sort_values(["case_order", "case_id"]).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x_values(sub), sub["share_same_full_decision"], marker="o")
    ax.set_title(f"Share of matches with same full decision as baseline — {family}")
    ax.set_xlabel("Case value")
    ax.set_ylabel("Share same full decision")
    ax.grid(True, alpha=0.3)
    savefig_if_needed(fig, f"Same_full_decision_share__{family}.png")
    plt.show()


In [ ]:

for family, sub in change_summary_df.groupby("case_family", dropna=False, observed=False):
    sub = sub.sort_values(["case_order", "case_id"]).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x_values(sub), sub["mean_seller_utility_diff_vs_baseline"], marker="o")
    ax.set_title(f"Mean seller-utility difference vs baseline — {family}")
    ax.set_xlabel("Case value")
    ax.set_ylabel("Mean seller utility difference")
    ax.grid(True, alpha=0.3)
    savefig_if_needed(fig, f"Seller_utility_diff__{family}.png")
    plt.show()


In [ ]:

if FILTERS["match_ids"] is not None and not compare_df.empty:
    plot_df = compare_df.sort_values(["match_id", "case_order", "case_id"]).copy()
    for match_id, sub in plot_df.groupby("match_id", dropna=False, observed=False):
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(pd.to_numeric(sub["case_order"], errors="coerce"), sub["seller_utility_diff_vs_baseline"], marker="o")
        ax.set_title(f"Seller utility difference vs baseline — match {match_id}")
        ax.set_xlabel("Case order")
        ax.set_ylabel("Seller utility difference")
        ax.grid(True, alpha=0.3)
        savefig_if_needed(fig, f"Seller_utility_diff__match_{int(match_id):03d}.png")
        plt.show()
else:
    print("Set FILTERS['match_ids'] to a list of match IDs if you want match-level traces.")
